# 6주차. 카나메이트, 세상에 나가다

**부제:** Supervisor/Sub-agent 위임과 Golden Scenario 하네스

## 0. 목표

이번 주에는 supervisor가 직접 일하지 않고, 요청 성격에 맞는 나나 또는 카나 sub-agent에게 위임하는 흐름을 검증한다. 실제 구현 문제는 별도 repo에서 다루고, 이 노트북은 라우팅 기준과 검증 payload를 정리한다.

성취 기준:

- supervisor와 sub-agent의 역할 차이를 설명할 수 있다.
- `selected_agent`와 내부 tool trace를 함께 확인할 수 있다.
- Golden Scenario가 라우팅 검증에 필요한 이유를 설명할 수 있다.

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

6주차의 실제 supervisor/sub-agent 구현 문제는 별도 문제 repo에서 작성한다. 이 레포에서는 노트북으로 routing contract, delegate payload, golden case 기준을 먼저 맞춘다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → sub-agent payload shape → golden case 검증 기준 순서로 진행한다.

In [ ]:
import json

def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))

golden_cases = [
    {
        "name": "personal_schedule",
        "request": "내일 9시에 민수와 1:1 일정 잡아줘",
        "member_replies": "",
        "expected_agent": "nana",
        "expected_inner_tool": "personal_create_schedule",
    },
    {
        "name": "group_slot",
        "request": "팀 회의 시간을 조율해줘",
        "member_replies": "민수: 2026-04-25 15:00 가능\n지아: 2026-04-25 15:00 가능",
        "expected_agent": "kana",
        "expected_inner_tool": "group_confirm_slot",
    },
]

show_json({"golden_cases": golden_cases})

## 2. 개념

오늘의 큰 그림: Sub-agent는 역할과 사용할 도구를 나눈 agent다. Supervisor는 직접 업무 도구를 들고 있지 않고, `nana_agent`, `kana_agent`처럼 sub-agent를 감싼 tool 중 하나를 호출한다.

반드시 이해할 것:

- supervisor trace는 어떤 sub-agent에게 위임했는지 보여준다.
- sub-agent 내부 trace는 실제 업무 tool이 실행됐는지 보여준다.
- `delegate_payload` 안에 sub-agent의 답변과 내부 trace가 들어 있다.
- Golden Scenario는 라우팅을 반복 가능한 케이스로 검증하는 장치다.

지금은 몰라도 되는 것:

- 대규모 multi-agent orchestration 패턴
- sub-agent별 장기 메모리 연결
- MCP와 sub-agent 통합

막혔을 때 볼 trace: supervisor의 `nana_agent`/`kana_agent` tool call, `delegate_payload.trace`, `inner_tool_names`, `passed`.

## 3. 기본 개념 실습

가장 작은 흐름은 supervisor가 개인 요청은 나나에게, 그룹 조율 요청은 카나에게 위임했다는 trace를 읽는 것이다.

In [ ]:
def fake_delegate_payload(agent: str, inner_tool: str) -> dict:
    return {
        "agent": agent,
        "answer": f"{agent}가 요청을 처리했습니다.",
        "trace": [
            {"event": "tool_call", "tool_name": inner_tool, "arguments": {}},
            {"event": "tool_result", "tool_name": inner_tool, "content": json.dumps({"ok": True}, ensure_ascii=False)},
        ],
    }

def fake_supervisor_trace(agent: str, delegate_payload: dict) -> list[dict]:
    tool_name = f"{agent}_agent"
    return [
        {"event": "tool_call", "tool_name": tool_name, "arguments": {}},
        {"event": "tool_result", "tool_name": tool_name, "content": json.dumps(delegate_payload, ensure_ascii=False)},
    ]

def inner_tool_names_from_payload(payload: dict) -> list[str]:
    return [event["tool_name"] for event in payload.get("trace", []) if event.get("event") == "tool_call"]

personal_payload = fake_delegate_payload("nana", "personal_create_schedule")
personal_trace = fake_supervisor_trace("nana", personal_payload)

show_json({"supervisor_trace": personal_trace, "delegate_payload": personal_payload})

## 4. 카나메이트 확장 예제

그룹 조율 케이스에서는 supervisor가 `kana_agent`를 호출하고, Kana 내부에서는 `group_confirm_slot` tool이 실행되어야 한다.

In [ ]:
group_payload = fake_delegate_payload("kana", "group_confirm_slot")
group_trace = fake_supervisor_trace("kana", group_payload)

show_json({
    "selected_agent": "kana",
    "inner_tool_names": inner_tool_names_from_payload(group_payload),
    "delegate_payload": group_payload,
    "supervisor_trace": group_trace,
})

## 5. 확장 예제 실행

Golden Scenario를 반복 실행한다. 최종 답변 문구가 조금 달라도 기대 agent와 내부 tool이 맞으면 통과로 본다.

In [ ]:
def evaluate_golden_case(case: dict) -> dict:
    selected_agent = "kana" if case.get("member_replies") else "nana"
    expected_tool = "group_confirm_slot" if selected_agent == "kana" else "personal_create_schedule"
    payload = fake_delegate_payload(selected_agent, expected_tool)
    inner_tool_names = inner_tool_names_from_payload(payload)
    return {
        "name": case["name"],
        "expected_agent": case["expected_agent"],
        "selected_agent": selected_agent,
        "expected_inner_tool": case["expected_inner_tool"],
        "inner_tool_names": inner_tool_names,
        "passed": case["expected_agent"] == selected_agent and case["expected_inner_tool"] in inner_tool_names,
        "delegate_payload": payload,
    }

reports = [evaluate_golden_case(case) for case in golden_cases]
assert all(report["passed"] for report in reports)

show_json(reports)

## 6. 회고

확인 질문:

1. supervisor가 직접 `personal_create_schedule`을 호출하지 않는 이유는 무엇인가?
2. `selected_agent`만 맞고 내부 tool이 틀리면 성공이라고 볼 수 있는가?
3. Golden Scenario는 눈으로 한 번 실행해 보는 것과 무엇이 다른가?
4. sub-agent 내부 trace를 supervisor 밖으로 꺼내 보여줘야 하는 이유는 무엇인가?

작은 응용 과제: 개인 일정처럼 보이지만 멤버 응답이 포함된 애매한 요청을 만들어 보고, 기대 agent와 내부 tool을 먼저 정한 뒤 trace 판정 기준을 작성한다.